# UCB Admissions — Aggregated Binomial & Simpson's Paradox

A classic example of **Simpson's paradox**: an association reverses direction when you condition on a third variable.

**Data**: UCB graduate admissions 1973 — applications, admissions, and gender across 6 departments.

**Estimand**: Does gender influence the probability of admission?

**Plan**:
1. Model 1 (gender only): estimate male vs female admission probability → contrast
2. Posterior check: reveals model failure — systematic pattern by department
3. Model 2 (gender + department): re-estimate contrast within departments
4. DAG: understand why the two models give opposite answers

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
from scipy import special

plt.style.use('default')
%matplotlib inline
rng = np.random.default_rng(42)

print(f'PyMC {pm.__version__} | ArviZ {az.__version__}')

## The Data

UCBadmit dataset: for each department (A–F) and gender, we have:
- `admit`: number of applicants admitted
- `applications`: total number of applicants
- `gender`: male or female

This is **aggregated binomial** data — each row is a count of successes out of N trials.

In [ ]:
url = 'https://raw.githubusercontent.com/rmcelreath/rethinking/master/data/UCBadmit.csv'
ucb = pd.read_csv(url, sep=';')
print(ucb.to_string())
print(f'\nShape: {ucb.shape}')
print(f'Columns: {ucb.columns.tolist()}')

In [ ]:
# Build index variables (0-indexed)
ucb['gender_id'] = (ucb['applicant.gender'] == 'male').astype(int)   # 0=female, 1=male
ucb['dept_id']   = pd.Categorical(ucb['dept']).codes                  # 0-5
ucb['admit_rate'] = ucb['admit'] / ucb['applications']

admit        = ucb['admit'].values
applications = ucb['applications'].values
gender_id    = ucb['gender_id'].values
dept_id      = ucb['dept_id'].values

n_depts   = ucb['dept_id'].nunique()
dept_labels   = ['A', 'B', 'C', 'D', 'E', 'F']
gender_labels = ['female', 'male']

print(ucb[['dept', 'applicant.gender', 'admit', 'applications', 'admit_rate', 'gender_id', 'dept_id']]
      .to_string(index=False))

# Aggregate admission rates by gender
print('\nOverall admission rates:')
print(ucb.groupby('applicant.gender').apply(
    lambda x: pd.Series({'admit_rate': x['admit'].sum() / x['applications'].sum(),
                         'total_apps': x['applications'].sum()})
).round(3).to_string())

In [ ]:
# Visualise raw admission rates by dept and gender
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: admission rates by department and gender
ax = axes[0]
male   = ucb[ucb['applicant.gender'] == 'male']
female = ucb[ucb['applicant.gender'] == 'female']
x = np.arange(n_depts)
ax.bar(x - 0.2, male['admit_rate'].values,   width=0.35, color='steelblue', alpha=0.8, label='male')
ax.bar(x + 0.2, female['admit_rate'].values, width=0.35, color='tomato',    alpha=0.8, label='female')
ax.set_xticks(x)
ax.set_xticklabels(dept_labels)
ax.set_xlabel('Department', fontsize=11)
ax.set_ylabel('Admission rate', fontsize=11)
ax.set_title('Admission rates by dept and gender\n(females competitive or better in most depts!)',
             fontsize=10, fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=0.3); ax.set_ylim(0, 1)

# Panel 2: application numbers by dept and gender
ax = axes[1]
ax.bar(x - 0.2, male['applications'].values,   width=0.35, color='steelblue', alpha=0.8, label='male')
ax.bar(x + 0.2, female['applications'].values, width=0.35, color='tomato',    alpha=0.8, label='female')
ax.set_xticks(x)
ax.set_xticklabels(dept_labels)
ax.set_xlabel('Department', fontsize=11)
ax.set_ylabel('Number of applications', fontsize=11)
ax.set_title('Application numbers by dept and gender\n(women apply more to competitive depts C-F!)',
             fontsize=10, fontweight='bold')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)

plt.suptitle('UCB Admissions 1973 — The Raw Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Model 1: Gender Only (m11.7)

Start simple — only gender as predictor:

$$A_i \sim \text{Binomial}(N_i, p_i)$$
$$\text{logit}(p_i) = \alpha_{\text{gender}[i]}$$
$$\alpha_{\text{gender}} \sim \text{Normal}(0, 1.5)$$

Two parameters: one intercept for female, one for male. No other predictors.

In [ ]:
coords = {
    'gender': gender_labels,
    'dept':   dept_labels,
}

with pm.Model(coords=coords) as m_gender:
    alpha = pm.Normal('alpha', mu=0, sigma=1.5, dims='gender')
    p     = pm.Deterministic('p', pm.math.invlogit(alpha[gender_id]))
    obs   = pm.Binomial('obs', n=applications, p=p, observed=admit)

    idata_gender = pm.sample(1000, tune=1000, chains=4,
                             target_accept=0.9, random_seed=42, progressbar=True)
    pm.compute_log_likelihood(idata_gender)

print(az.summary(idata_gender, var_names=['alpha'], hdi_prob=0.89).to_string())

In [ ]:
# Compute contrast: male - female admission probability
post_g = idata_gender.posterior
alpha_s = post_g['alpha'].values.reshape(-1, 2)   # (S, 2): col0=female, col1=male

# On log-odds scale
diff_a = alpha_s[:, 1] - alpha_s[:, 0]            # male - female (log-odds)

# On probability scale
diff_p = special.expit(alpha_s[:, 1]) - special.expit(alpha_s[:, 0])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
az.plot_posterior({'diff (log-odds)': diff_a}, hdi_prob=0.89, ref_val=0, ax=ax, color='steelblue')
ax.set_title('Male − Female contrast\n(log-odds scale)', fontsize=11, fontweight='bold')

ax = axes[1]
az.plot_posterior({'diff (probability)': diff_p}, hdi_prob=0.89, ref_val=0, ax=ax, color='tomato')
ax.set_title('Male − Female contrast\n(probability scale)', fontsize=11, fontweight='bold')

plt.suptitle('Model 1 (gender only): Is there a male admission advantage?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Male advantage (log-odds): mean={diff_a.mean():+.3f},  '
      f'89% CI [{np.percentile(diff_a,5.5):+.3f}, {np.percentile(diff_a,94.5):+.3f}]')
print(f'Male advantage (prob):     mean={diff_p.mean():+.3f},  '
      f'89% CI [{np.percentile(diff_p,5.5):+.3f}, {np.percentile(diff_p,94.5):+.3f}]')
print(f'P(male advantage) = {(diff_p > 0).mean():.1%}')

## Posterior Validation Check

The model says males have ~14% higher admission probability. But is the model fitting well?

For each row in the data, compute the **expected admission rate** from the model and compare to the **observed rate**. If the model is good, points should scatter around the diagonal.

**Key**: even though department is NOT in the model, we can still group the residuals by department to see if there is a systematic pattern.

> **Your question**: *how can we connect lines from the same dept if dept was not in regression?*
>
> The model predicts the same `p` for ALL males (regardless of dept) and the same `p` for ALL females. So the "expected" values are just two horizontal lines. We then overlay the observed rates per dept-gender combination and connect the two gender points within each department. The lines reveal that the model's one-size-fits-all prediction doesn't match any individual department — a clear sign of a missing variable.

In [ ]:
# Posterior predicted admission rate for each row
p_s = post_g['p'].values.reshape(-1, len(ucb))   # (S, 12)
p_mean = p_s.mean(axis=0)                         # posterior mean per row
p_lo   = np.percentile(p_s, 5.5,  axis=0)
p_hi   = np.percentile(p_s, 94.5, axis=0)

fig, ax = plt.subplots(figsize=(9, 6))

colors_dept = plt.cm.Set2(np.linspace(0, 1, n_depts))

# Plot each row: observed vs predicted
for i, row in ucb.iterrows():
    color = colors_dept[int(row['dept_id'])]
    marker = 'o' if row['applicant.gender'] == 'male' else 's'
    ax.plot(i+1, row['admit_rate'], marker=marker, color=color, ms=9, zorder=5)
    ax.errorbar(i+1, p_mean[i], yerr=[[p_mean[i]-p_lo[i]], [p_hi[i]-p_mean[i]]],
                fmt='+', color=color, ms=9, capsize=4, lw=1.5)

# Connect same-dept points (pairs: male row and female row for each dept)
for d in range(n_depts):
    dept_rows = ucb[ucb['dept_id'] == d].index.tolist()
    if len(dept_rows) == 2:
        # Connect observed points
        y1, y2 = ucb.loc[dept_rows[0], 'admit_rate'], ucb.loc[dept_rows[1], 'admit_rate']
        ax.plot([dept_rows[0]+1, dept_rows[1]+1], [y1, y2],
                color=colors_dept[d], lw=1.5, alpha=0.6)
        # Connect predicted points
        ax.plot([dept_rows[0]+1, dept_rows[1]+1], [p_mean[dept_rows[0]], p_mean[dept_rows[1]]],
                color=colors_dept[d], lw=1.5, ls='--', alpha=0.6)

# Legend
from matplotlib.lines import Line2D
legend_elements = ([Line2D([0],[0], marker='o', color='gray', label='male (obs)', ms=8, ls='none'),
                    Line2D([0],[0], marker='s', color='gray', label='female (obs)', ms=8, ls='none'),
                    Line2D([0],[0], marker='+', color='gray', label='predicted (89% CI)', ms=8)] +
                   [Line2D([0],[0], color=colors_dept[d], label=f'Dept {dept_labels[d]}', lw=3)
                    for d in range(n_depts)])
ax.legend(handles=legend_elements, fontsize=8, loc='upper right', ncol=2)

ax.set_xticks(range(1, len(ucb)+1))
ax.set_xticklabels([f"{row['dept']}\n{row['applicant.gender'][:1].upper()}"
                    for _, row in ucb.iterrows()], fontsize=8)
ax.set_ylabel('Admission rate', fontsize=11)
ax.set_ylim(0, 1)
ax.axhline(0.5, color='gray', ls=':', alpha=0.4)
ax.grid(True, axis='y', alpha=0.3)
ax.set_title('Posterior validation check — Model 1 (gender only)\n'
             'Solid lines = observed, dashed = predicted. '
             'Model predicts same p for all males/females — systematic dept pattern missed!',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## Model 2: Gender + Department (m11.8)

The posterior check reveals a systematic pattern: departments cluster together. Some departments (A, B) have high admission rates overall; others (F) are very competitive. Women apply more to the competitive departments.

Add department as an index variable:

$$A_i \sim \text{Binomial}(N_i, p_i)$$
$$\text{logit}(p_i) = \alpha_{\text{gender}[i]} + \delta_{\text{dept}[i]}$$
$$\alpha_{\text{gender}} \sim \text{Normal}(0, 1.5)$$
$$\delta_{\text{dept}} \sim \text{Normal}(0, 1.5)$$

Now each department gets its own baseline admission rate, and `α[gender]` captures the gender effect **within departments**.

In [ ]:
with pm.Model(coords=coords) as m_dept_gender:
    alpha = pm.Normal('alpha', mu=0, sigma=1.5, dims='gender')
    delta = pm.Normal('delta', mu=0, sigma=1.5, dims='dept')

    logit_p = alpha[gender_id] + delta[dept_id]
    p = pm.Deterministic('p', pm.math.invlogit(logit_p))
    obs = pm.Binomial('obs', n=applications, p=p, observed=admit)

    idata_dept = pm.sample(1000, tune=1000, chains=4,
                           target_accept=0.9, random_seed=42, progressbar=True)
    pm.compute_log_likelihood(idata_dept)

print(az.summary(idata_dept, var_names=['alpha', 'delta'], hdi_prob=0.89).to_string())

In [ ]:
# Contrast: male - female, now controlling for department
post_d  = idata_dept.posterior
alpha_d = post_d['alpha'].values.reshape(-1, 2)    # (S, 2): col0=female, col1=male

diff_a2 = alpha_d[:, 1] - alpha_d[:, 0]            # log-odds
diff_p2 = special.expit(alpha_d[:, 1]) - special.expit(alpha_d[:, 0])  # probability

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
az.plot_posterior({'diff (log-odds)': diff_a2}, hdi_prob=0.89, ref_val=0, ax=ax, color='steelblue')
ax.set_title('Male − Female contrast\n(log-odds, controlling for dept)', fontsize=11, fontweight='bold')

ax = axes[1]
az.plot_posterior({'diff (probability)': diff_p2}, hdi_prob=0.89, ref_val=0, ax=ax, color='tomato')
ax.set_title('Male − Female contrast\n(probability, controlling for dept)', fontsize=11, fontweight='bold')

plt.suptitle('Model 2 (gender + dept): contrast reverses direction!',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Model 1 (gender only):     male advantage = {diff_p.mean():+.3f}  (males favoured)')
print(f'Model 2 (gender + dept):   male advantage = {diff_p2.mean():+.3f}  (females slightly favoured within depts)')
print()
print('The direction REVERSED — this is Simpson\'s paradox.')

In [ ]:
# Side-by-side comparison of the two contrasts
fig, ax = plt.subplots(figsize=(9, 4))

for contrast, label, color, y in [
    (diff_p,  'Model 1: gender only\n(no dept control)',    'tomato',    1),
    (diff_p2, 'Model 2: gender + dept\n(controls for dept)', 'steelblue', 0),
]:
    lo, mid, hi = np.percentile(contrast, [5.5, 50, 94.5])
    ax.plot([lo, hi], [y, y], color=color, lw=5, alpha=0.8)
    ax.plot(mid, y, 'o', color=color, ms=10, zorder=5)
    ax.text(mid + 0.01, y + 0.08, f'{mid:+.3f}', fontsize=10, color=color, fontweight='bold')

ax.axvline(0, color='black', ls='--', lw=1.5, alpha=0.6, label='no difference')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Model 2\n(gender + dept)', 'Model 1\n(gender only)'], fontsize=10)
ax.set_xlabel('Male − Female admission probability', fontsize=11)
ax.set_xlim(-0.15, 0.25)
ax.set_title("Simpson's Paradox in UCB Admissions\n"
             "Male advantage disappears (reverses) once you control for department",
             fontsize=11, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Understanding the DAG

Why do the two models disagree? The causal structure explains everything.

```
G ──────────────────→ A
 ╲                  ↗
  ╲→ D ────────────
```

- **G** = Gender  
- **D** = Department applied to  
- **A** = Admission decision

Two paths from G to A:
1. **Direct path**: G → A — gender directly influences admission within a department
2. **Indirect path**: G → D → A — gender influences *which department* you apply to, and departments have different admission rates

### What each model estimates:

| Model | What it estimates |
|-------|------------------|
| Gender only (m1) | **Total effect** of gender = direct + indirect path |
| Gender + dept (m2) | **Direct effect** of gender (blocks G → D → A by conditioning on D) |

### Why they contradict:

- Women disproportionately applied to **competitive departments** (C, D, E, F — low admission rates)
- Men disproportionately applied to **easy departments** (A, B — high admission rates)
- So in the aggregate, men appear to be admitted more — but this is because of *where they applied*, not because of bias
- Within each department, women are admitted at similar or slightly higher rates

### Which model answers the right question?

It depends on the **estimand**:
- If you want to know "is there direct discrimination within departments?" → Model 2 (control for D)
- If you want to know "does being female harm your overall admission chances?" → Model 1 (total effect)
- D is a **mediator** on the G → D → A path — conditioning on it blocks the indirect effect

> Note: This is a **mediation** example, not a confounding example. Department is caused partly by gender (G → D), so it is a mediator, not a confounder. Conditioning on a mediator is appropriate when you want only the direct effect.

In [ ]:
# Draw the DAG
fig, ax = plt.subplots(figsize=(7, 4))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

# Nodes
nodes = {'G': (1, 3), 'D': (5, 1), 'A': (9, 3)}
for name, (x, y) in nodes.items():
    ax.add_patch(plt.Circle((x, y), 0.6, color='steelblue', alpha=0.3, zorder=2))
    ax.text(x, y, name, ha='center', va='center', fontsize=16, fontweight='bold', zorder=3)

# Arrows
arrow_kw = dict(arrowstyle='->', color='black', lw=2)
# G → A (direct)
ax.annotate('', xy=(8.4, 3), xytext=(1.6, 3),
            arrowprops=dict(**arrow_kw))
ax.text(5, 3.4, 'direct effect', ha='center', fontsize=9, style='italic', color='steelblue')
# G → D
ax.annotate('', xy=(4.4, 1.3), xytext=(1.5, 2.6),
            arrowprops=dict(**arrow_kw))
# D → A
ax.annotate('', xy=(8.4, 2.7), xytext=(5.6, 1.3),
            arrowprops=dict(**arrow_kw))
ax.text(5, 0.4, 'G → D → A  (indirect / mediation path)', ha='center',
        fontsize=9, style='italic', color='tomato')

ax.set_title('DAG: Gender (G), Department (D), Admission (A)\n'
             'D is a MEDIATOR — conditioning on it blocks the indirect path',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Department admission rates (posterior from model 2)
delta_s = post_d['delta'].values.reshape(-1, n_depts)   # (S, 6)
dept_p  = special.expit(delta_s)   # baseline admission prob per dept (at average gender)

fig, ax = plt.subplots(figsize=(9, 4))
colors_dept = plt.cm.Set2(np.linspace(0, 1, n_depts))
for d in range(n_depts):
    lo, mid, hi = np.percentile(dept_p[:, d], [5.5, 50, 94.5])
    ax.plot([lo, hi], [d, d], color=colors_dept[d], lw=5, alpha=0.8)
    ax.plot(mid, d, 'o', color=colors_dept[d], ms=10, zorder=5)
    # also show proportion of female applicants
    dept_data  = ucb[ucb['dept_id'] == d]
    f_apps     = dept_data[dept_data['applicant.gender']=='female']['applications'].values[0]
    total_apps = dept_data['applications'].sum()
    ax.text(mid + 0.01, d + 0.15, f'  {mid:.2f}  ({f_apps/total_apps:.0%} female applicants)',
            fontsize=9, va='bottom')

ax.set_yticks(range(n_depts))
ax.set_yticklabels([f'Dept {l}' for l in dept_labels], fontsize=10)
ax.set_xlabel('Baseline admission probability', fontsize=11)
ax.set_title('Department admission rates (Model 2) with % female applicants\n'
             'Hard depts (E, F) attract more female applicants → drives apparent male advantage',
             fontsize=10, fontweight='bold')
ax.set_xlim(0, 1)
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## Key Insights

### 1. Simpson's Paradox
An association in the aggregate **reverses** when you condition on a subgroup. Here: men appear advantaged overall, but within every department women are equally or slightly better admitted.

### 2. The posterior validation check revealed the problem
Even without department in the model, plotting residuals grouped by department showed a clear systematic pattern — the model predicted the same rate for all males and all females regardless of department, which didn't match any department's actual rates.

### 3. Department is a mediator, not a confounder
- **Confounder**: causes both treatment and outcome → must condition to remove bias
- **Mediator**: caused by treatment, causes outcome → conditioning blocks the indirect effect
- The right choice depends on the **estimand**: total effect or direct effect?

### 4. Aggregated binomial is identical to disaggregated Bernoulli
We used one row per dept-gender combination with `Binomial(n, p)`. Could have used one row per applicant with `Bernoulli(p)`. Same posterior.

---
*Chapter 11 — Statistical Rethinking (McElreath, 2nd ed.)*